# Lesson 2.3 — Writing Your First Template

**Phase 2: Chart Anatomy & Templates**

---

## Learning Objectives

By the end of this lesson you will be able to:
- Use `{{ .Values.x }}` and `{{ .Release.Name }}` in templates
- Write a Deployment and Service template using values
- Render and validate templates with `helm template`
- Understand template delimiters and whitespace control

---

## Template Basics

Helm templates are standard Kubernetes YAML files with **Go template directives** embedded inside `{{ }}` blocks.

### The Dot (`.`) — Current Scope

The dot `.` refers to the *top-level data object* passed to the template. It contains:

| Object | Contents |
|---|---|
| `.Values` | All values from `values.yaml` and user overrides |
| `.Release` | Info about the current release (Name, Namespace, etc.) |
| `.Chart` | Contents of `Chart.yaml` (Name, Version, AppVersion) |
| `.Files` | Non-template files bundled in the chart |
| `.Capabilities` | Info about the Kubernetes cluster |

> **Reference:** [Chart Template Guide](https://helm.sh/docs/chart_template_guide/) — https://helm.sh/docs/chart_template_guide/

---

## Set Up a Working Chart

In [ ]:
# Create a clean chart for this lesson
rm -rf /tmp/templatedemo && mkdir -p /tmp/templatedemo/templates

cat > /tmp/templatedemo/Chart.yaml << 'EOF'
apiVersion: v2
name: templatedemo
version: 0.1.0
appVersion: "1.0.0"
EOF

cat > /tmp/templatedemo/values.yaml << 'EOF'
replicaCount: 1

image:
  repository: nginx
  tag: "1.25"
  pullPolicy: IfNotPresent

service:
  type: ClusterIP
  port: 80

containerPort: 80
EOF

echo "Chart scaffolded"

---

## Writing a Deployment Template

In [ ]:
%%writefile /tmp/templatedemo/templates/deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  # {{ .Release.Name }} is the name given at helm install time
  name: {{ .Release.Name }}-{{ .Chart.Name }}
  namespace: {{ .Release.Namespace }}
  labels:
    app: {{ .Chart.Name }}
    release: {{ .Release.Name }}
    version: {{ .Chart.AppVersion }}
spec:
  # .Values.replicaCount comes from values.yaml (or user override)
  replicas: {{ .Values.replicaCount }}
  selector:
    matchLabels:
      app: {{ .Chart.Name }}
      release: {{ .Release.Name }}
  template:
    metadata:
      labels:
        app: {{ .Chart.Name }}
        release: {{ .Release.Name }}
    spec:
      containers:
        - name: {{ .Chart.Name }}
          # Concatenate repository and tag with a colon
          image: {{ .Values.image.repository }}:{{ .Values.image.tag }}
          imagePullPolicy: {{ .Values.image.pullPolicy }}
          ports:
            - containerPort: {{ .Values.containerPort }}

In [ ]:
# Render the template and see the output
helm template my-release /tmp/templatedemo

---

## Writing a Service Template

In [ ]:
%%writefile /tmp/templatedemo/templates/service.yaml
apiVersion: v1
kind: Service
metadata:
  name: {{ .Release.Name }}-{{ .Chart.Name }}
  namespace: {{ .Release.Namespace }}
spec:
  type: {{ .Values.service.type }}
  ports:
    - port: {{ .Values.service.port }}
      targetPort: {{ .Values.containerPort }}
      protocol: TCP
  selector:
    app: {{ .Chart.Name }}
    release: {{ .Release.Name }}

In [ ]:
# Render again — now we get both Deployment and Service
helm template my-release /tmp/templatedemo

---

## Whitespace Control

Go templates can leave unwanted blank lines. Use `-` to trim whitespace:

| Syntax | Trims |
|---|---|
| `{{- ` | whitespace/newline **before** the tag |
| ` -}}` | whitespace/newline **after** the tag |
| `{{- ... -}}` | both sides |

In [ ]:
# Create a test template showing whitespace control
cat > /tmp/templatedemo/templates/whitespace-demo.yaml << 'EOF'
apiVersion: v1
kind: ConfigMap
metadata:
  name: whitespace-demo
data:
  # Without trimming — notice blank line above 'release:'
  releaseName: {{ .Release.Name }}
  # With left-trim — removes the leading newline
  chartName: {{- .Chart.Name }}
EOF

helm template my-release /tmp/templatedemo --show-only templates/whitespace-demo.yaml

---

## Quoting Values Safely

String values in YAML can cause parse errors if not quoted. Use the `quote` pipeline function:

In [ ]:
cat > /tmp/templatedemo/templates/quoting-demo.yaml << 'EOF'
apiVersion: v1
kind: ConfigMap
metadata:
  name: quoting-demo
data:
  # .Values.image.tag might be '1.25' or 'latest' — quote it to be safe
  imageTag: {{ .Values.image.tag | quote }}

  # Integers don't need quoting but strings do
  port: {{ .Values.service.port }}
  portAsString: {{ .Values.service.port | quote }}
EOF

helm template my-release /tmp/templatedemo --show-only templates/quoting-demo.yaml

---

## Verifying a Specific Template

Use `--show-only` to render a single template file:

In [ ]:
# Render only the deployment template
helm template my-release /tmp/templatedemo \
  --show-only templates/deployment.yaml \
  --set replicaCount=3 \
  --set image.tag=1.26

---

## Exercise

In [ ]:
# TASK: Add a ConfigMap template to /tmp/templatedemo/templates/configmap.yaml
# that creates a ConfigMap with:
#   - name: <release-name>-config
#   - data keys: env (from .Values.env, default value should be 'development'),
#                appVersion (from .Chart.AppVersion)
#
# Steps:
# 1. Add 'env: development' to /tmp/templatedemo/values.yaml
# 2. Create the ConfigMap template
# 3. Render the chart with helm template and verify the output
# 4. Override env to 'production' using --set and verify the change

# TODO: write your implementation here

echo "Exercise: implement the steps above"

---

## Summary

- Templates are YAML files with `{{ }}` Go-template directives.
- `.Values`, `.Release`, `.Chart`, `.Files`, `.Capabilities` are the top-level objects available in every template.
- Use `{{- }}` and `{{ -}}` to strip unwanted whitespace.
- Use `| quote` to safely wrap string values.
- `helm template --show-only` lets you render and inspect a single template file.

---

## Next Lesson

➡️ [Lesson 2.4 — Built-in Objects & Rendering](lesson-2.4-builtin-objects.ipynb)